In [75]:
import yfinance as yf
import pandas as pd
import numpy as np
import json
from IPython.display import display

def momentum_screener(tickers, threshold_json='thresholds.json'):
    """
    Complete momentum analysis from scratch - fetches data, calculates metrics, and applies threshold-based coloring.
    
    Parameters:
    -----------
    tickers : list
        List of ticker symbols to analyze
    threshold_json : str
        Path to JSON file containing threshold definitions
    
    Returns:
    --------
    DataFrame with calculated metrics and styled output
    """
    
    # Load thresholds from JSON
    with open(threshold_json, 'r') as f:
        thresholds = json.load(f)
        
    # Fetch benchmark data (VOO) for RS comparison
    print("Fetching benchmark data (VOO)...")
    voo_close = None
    try:
        voo_data = yf.download('VOO', period='max', progress=False)
        
        # Handle MultiIndex for VOO
        if isinstance(voo_data.columns, pd.MultiIndex):
            try:
                if 'VOO' in voo_data.columns.get_level_values(1):
                    voo_data = voo_data.xs('VOO', axis=1, level=1)
                else:
                    voo_data.columns = voo_data.columns.droplevel(1)
            except:
                pass
        
        # Extract Close column
        if 'Close' in voo_data.columns:
             temp_close = voo_data['Close']
             if isinstance(temp_close, pd.DataFrame):
                 voo_close = temp_close.iloc[:, 0]
             else:
                 voo_close = temp_close
    except Exception as e:
        print(f"  Warning: Failed to download VOO data: {e}")
    
    results = []
    
    from sklearn.linear_model import LinearRegression
    
    for ticker in tickers:
        try:
            # Download data
            data = yf.download(ticker, period='max', progress=False)
            
            if data.empty or len(data) < 200:
                print(f"  Insufficient data for {ticker}")
                continue
            
            # Handle MultiIndex columns (common in new yfinance)
            if isinstance(data.columns, pd.MultiIndex):
                try:
                    if ticker in data.columns.get_level_values(1):
                         data = data.xs(ticker, axis=1, level=1)
                    else:
                         data.columns = data.columns.droplevel(1)
                except Exception as e:
                     print(f"  Warning: Could not handle MultiIndex for {ticker}: {e}")
            
            if isinstance(data, pd.DataFrame) and 'Close' in data.columns:
                close_col = data['Close']
                if isinstance(close_col, pd.DataFrame):
                    print(f"  Warning: Duplicate 'Close' columns found for {ticker}, using first one.")
                    data = data.loc[:, ~data.columns.duplicated()]
            
            current_price = data['Close'].iloc[-1]
            returns = data['Close'].pct_change()
            
            # Calculate ratio series (ticker/VOO)
            ratio_series = None
            if voo_close is not None:
                try:
                    t_series = data['Close']
                    if isinstance(t_series, pd.DataFrame):
                        t_series = t_series.iloc[:, 0]
                    v_series = voo_close
                    if isinstance(v_series, pd.DataFrame):
                        v_series = v_series.iloc[:, 0]
                    aligned = pd.concat([t_series, v_series], axis=1, join='inner').dropna()
                    ratio_series = aligned.iloc[:, 0] / aligned.iloc[:, 1]
                except Exception:
                    ratio_series = None
            
            # CAGR calculation
            cagr = np.nan
            if len(data['Close']) > 1:
                start_price = data['Close'].iloc[0]
                end_price = data['Close'].iloc[-1]
                n_days = (data.index[-1] - data.index[0]).days
                n_years = n_days / 365.25 if n_days > 0 else np.nan
                if start_price > 0 and n_years > 0:
                    cagr = (end_price / start_price) ** (1 / n_years) - 1
                    cagr = cagr * 100
            
            # Distribution Days: high volume and price decline (no uptrend required)
            distribution_days = np.nan
            if 'Volume' in data.columns and 'Close' in data.columns:
                try:
                    # Distribution day: price down > 0.5% and volume > 1.2x 50d avg
                    price_down = data['Close'].pct_change() < -0.005
                    avg_vol_50 = data['Volume'].rolling(50, min_periods=20).mean()
                    high_vol = data['Volume'] > (avg_vol_50 * 1.2)
                    dist_day_mask = price_down & high_vol
                    distribution_days = dist_day_mask.iloc[-30:].sum()  # Last 30 days
                except Exception:
                    distribution_days = np.nan
            
            # RS vs VOO (6M)
            rs_vs_voo = np.nan
            if ratio_series is not None and len(ratio_series) >= 126:
                try:
                    rs_vs_voo = ((ratio_series.iloc[-1] / ratio_series.shift(126).iloc[-1]) - 1) * 100
                except Exception as e:
                    pass
            # RS vs VOO (12M) ex1 (exclude most recent month, 20 days)
            rs_vs_voo_12m_ex1 = np.nan
            if ratio_series is not None and len(ratio_series) >= 252 + 20:
                try:
                    # Compare current value to value 252 trading days ago, but exclude last 21 days
                    ref_idx = -21  # -1 is last, so -22 is 21 days before last
                    rs_vs_voo_12m_ex1 = ((ratio_series.iloc[ref_idx] / ratio_series.shift(252).iloc[ref_idx]) - 1) * 100
                except Exception as e:
                    pass
            
            # RS Slope (Linear Regression, 6M ex 1M: start 7M ago, end 1M ago)
            rs_slope_6m_ex1m = np.nan
            window_len = 126  # 6 months
            exclude_recent = 21  # 1 month
            if ratio_series is not None and len(ratio_series.dropna()) >= window_len + exclude_recent:
                try:
                    # Slice from 7M ago to 1M ago
                    start_idx = -(window_len + exclude_recent)
                    end_idx = -exclude_recent
                    y_rs = ratio_series.dropna().iloc[start_idx:end_idx].values.reshape(-1,1)
                    x_rs = np.arange(len(y_rs)).reshape(-1,1)
                    model_rs = LinearRegression().fit(x_rs, y_rs)
                    slope_rs = model_rs.coef_[0][0]
                    rs_slope_6m_ex1m = slope_rs / y_rs[-1][0] * 100 if y_rs[-1][0] != 0 else np.nan  # Remove 100x adjustment
                except Exception as e:
                    rs_slope_6m_ex1m = np.nan
            
            # RS Volatility (6M, annualized, end 1M ago)
            rs_vol_6m_ex1m = np.nan
            if ratio_series is not None and len(ratio_series.dropna()) >= window_len + exclude_recent:
                try:
                    # Use same window as RS Slope: from 7M ago to 1M ago
                    start_idx = -(window_len + exclude_recent)
                    end_idx = -exclude_recent
                    rs_window = ratio_series.dropna().iloc[start_idx:end_idx]
                    rs_returns = rs_window.pct_change().dropna()
                    if len(rs_returns) > 1:
                        rs_vol_6m_ex1m = rs_returns.std() * np.sqrt(252) * 100
                except Exception as e:
                    rs_vol_6m_ex1m = np.nan
            
            # RS Volatility (12M ex1, annualized, end 1M ago)
            rs_vol_12m_ex1m = np.nan
            window_len_12m = 252  # 12 months
            if ratio_series is not None and len(ratio_series.dropna()) >= window_len_12m + exclude_recent:
                try:
                    # From 13M ago to 1M ago
                    start_idx = -(window_len_12m + exclude_recent)
                    end_idx = -exclude_recent
                    rs_window_12m = ratio_series.dropna().iloc[start_idx:end_idx]
                    rs_returns_12m = rs_window_12m.pct_change().dropna()
                    if len(rs_returns_12m) > 1:
                        rs_vol_12m_ex1m = rs_returns_12m.std() * np.sqrt(252) * 100
                except Exception as e:
                    rs_vol_12m_ex1m = np.nan
            
            # General Volatility 1Y (annualized, using price, not ratio)
            volatility_1y = np.nan
            if len(data['Close'].dropna()) >= 252:
                try:
                    close_1y = data['Close'].dropna().iloc[-252:]
                    returns_1y = close_1y.pct_change().dropna()
                    if len(returns_1y) > 1:
                        volatility_1y = returns_1y.std() * np.sqrt(252) * 100
                except Exception as e:
                    volatility_1y = np.nan
            
            # Moving averages
            if ratio_series is not None and len(ratio_series) >= 200:
                sma_50 = ratio_series.rolling(50, min_periods=50).mean().iloc[-1]
                sma_200_series = ratio_series.rolling(200, min_periods=200).mean()
                sma_200 = sma_200_series.iloc[-1]
            else:
                sma_50 = np.nan
                sma_200 = np.nan
                sma_200_series = None
            
            # Ratio vs ratio SMA metrics
            if ratio_series is not None and np.isfinite(sma_50) and np.isfinite(sma_200):
                current_ratio = ratio_series.iloc[-1]
                sma_50_val = ratio_series.rolling(50, min_periods=50).mean().iloc[-1]
                sma_200_val = sma_200
                ratio_vs_sma50  = (current_ratio / sma_50_val  - 1) * 100
                ratio_vs_sma200 = (current_ratio / sma_200_val - 1) * 100
                ratio_above_sma200 = current_ratio > sma_200_val
                sma50_above_sma200 = sma_50_val > sma_200_val
                # Ratio SMA200 slope (126d, ~6 months)
                sma200_slope_window = 126
                if sma_200_series is not None and len(sma_200_series.dropna()) >= sma200_slope_window:
                    y = sma_200_series.iloc[-sma200_slope_window:].values.reshape(-1,1)
                    x = np.arange(len(y)).reshape(-1,1)
                    model = LinearRegression().fit(x, y)
                    slope = model.coef_[0][0]
                    slope_pct = slope / sma_200_series.iloc[-1] * 100 if sma_200_series.iloc[-1] != 0 else np.nan  # Remove 100x adjustment
                    ratio_sma200_slope_126d = slope_pct
                else:
                    ratio_sma200_slope_126d = np.nan
                # OBV calculation
                obv = None
                if 'Volume' in data.columns:
                    volume = data['Volume']
                    price = data['Close']
                    direction = np.sign(price.diff().fillna(0))
                    obv = (direction * volume).cumsum()
                # OBV Slope (100d, linear regression)
                obv_slope_100d = np.nan
                if obv is not None and len(obv.dropna()) >= 100:
                    try:
                        y_obv = obv.dropna().iloc[-100:].values.reshape(-1,1)
                        x_obv = np.arange(len(y_obv)).reshape(-1,1)
                        model_obv = LinearRegression().fit(x_obv, y_obv)
                        slope_obv = model_obv.coef_[0][0]
                        obv_slope_100d = slope_obv / y_obv[-1][0] * 100 if y_obv[-1][0] != 0 else np.nan  # Remove 100x adjustment
                    except Exception as e:
                        obv_slope_100d = np.nan
                # Max Price Drawdown (6mo, %), reported as positive value
                max_drawdown_6mo = np.nan
                if len(data['Close'].dropna()) >= 126:
                    window = 126
                    close_6mo = data['Close'].dropna().iloc[-window:]
                    rolling_max = close_6mo.cummax()
                    drawdown = (close_6mo - rolling_max) / rolling_max * 100
                    max_drawdown_6mo = -drawdown.min()  # Remove negative, report as positive % loss
                # RVOL and Persistent RVOL
                persistent_rvol_60d = np.nan  # Always define before any conditional
                if 'Volume' in data.columns:
                    volume = data['Volume']
                    # Require at least 10-15 days for a meaningful volume average
                    # Or use min_periods=20 to only calculate when you have a full period
                    sma20_vol = volume.rolling(20, min_periods=10).mean()
                    # Avoid division by zero or NaN
                    valid_mask = (sma20_vol > 0) & (~sma20_vol.isna()) & (volume > 0)  # Also check current volume
                    rvol_series = pd.Series(np.nan, index=volume.index)
                    rvol_series[valid_mask] = volume[valid_mask] / sma20_vol[valid_mask]
                    if len(rvol_series.dropna()) > 0:
                        # Persistent RVOL: % of last 60 days where RVOL > 1.2
                        last_60_rvol = rvol_series.iloc[-60:]
                        valid_last_60 = last_60_rvol.dropna()
                        persistent_rvol_60d = (valid_last_60 > 1.2).mean() * 100 if len(valid_last_60) > 0 else np.nan
                # UVP (Up/Down Volume Pressure, 20d and 60d normalized)
                uvp_20d = np.nan
                uvp_60d = np.nan
                if {'Volume', 'Close'}.issubset(data.columns):
                    close = data['Close']
                    volume = data['Volume']
                    up_mask = close.diff() > 0
                    down_mask = close.diff() < 0
                    # 20d UVP
                    up_vol_20 = volume.where(up_mask, 0.0).rolling(20, min_periods=20).sum()
                    down_vol_20 = volume.where(down_mask, 0.0).rolling(20, min_periods=20).sum()
                    total_vol_20 = volume.rolling(20, min_periods=20).sum()
                    uvp_norm_20 = (up_vol_20 - down_vol_20) / total_vol_20 * 100  # Multiply by 100
                    uvp_norm_20_valid = uvp_norm_20.dropna()
                    if not uvp_norm_20_valid.empty:
                        uvp_20d = float(uvp_norm_20_valid.iloc[-1])
                    # 60d UVP
                    up_vol_60 = volume.where(up_mask, 0.0).rolling(60, min_periods=60).sum()
                    down_vol_60 = volume.where(down_mask, 0.0).rolling(60, min_periods=60).sum()
                    total_vol_60 = volume.rolling(60, min_periods=60).sum()
                    uvp_norm_60 = (up_vol_60 - down_vol_60) / total_vol_60 * 100  # Multiply by 100
                    uvp_norm_60_valid = uvp_norm_60.dropna()
                    if not uvp_norm_60_valid.empty:
                        uvp_60d = float(uvp_norm_60_valid.iloc[-1])
                # Volatility Ratio (20d/100d)
                volatility_ratio = np.nan
                if 'Volume' in data.columns:
                    volume = data['Volume']
                    vol_20d = volume.rolling(20, min_periods=20).std()
                    vol_100d = volume.rolling(100, min_periods=100).std()
                    if len(vol_20d) > 0 and len(vol_100d) > 0 and np.isfinite(vol_20d.iloc[-1]) and np.isfinite(vol_100d.iloc[-1]) and vol_100d.iloc[-1] != 0:
                        volatility_ratio = vol_20d.iloc[-1] / vol_100d.iloc[-1]
            else:
                ratio_vs_sma50 = ratio_vs_sma200 = np.nan
                ratio_above_sma200 = sma50_above_sma200 = False
                ratio_sma200_slope_126d = np.nan
                obv_slope_100d = np.nan
                max_drawdown_6mo = np.nan
                persistent_rvol_60d = np.nan
                uvp_20d = np.nan
                uvp_60d = np.nan
                volatility_ratio = np.nan
            
            # % of last 6 months ratio above SMA200
            pct_above_sma200_6mo = np.nan
            if ratio_series is not None and sma_200_series is not None and len(ratio_series) >= 126:
                last_6mo = min(126, len(ratio_series))
                valid = sma_200_series[-last_6mo:].notna()
                if valid.any():
                    pct_above_sma200_6mo = (ratio_series[-last_6mo:][valid] > sma_200_series[-last_6mo:][valid]).mean() * 100
            
            # RSI calculation
            rsi_current = np.nan
            rsi_series = None
            relative_pressure_balance = np.nan
            momentum_efficiency_slope = np.nan
            rsi_slope_20d = np.nan
            if ratio_series is not None and len(ratio_series) >= 21:
                ratio_delta = ratio_series - ratio_series.shift(1)
                gain = ratio_delta.where(ratio_delta > 0, 0)
                loss = -ratio_delta.where(ratio_delta < 0, 0)
                avg_gain = gain.rolling(window=14, min_periods=14).mean()
                avg_loss = loss.rolling(window=14, min_periods=14).mean()
                rs = avg_gain / avg_loss
                rsi = 100 - (100 / (1 + rs))
                rsi_current = rsi.iloc[-1]
                rsi_series = rsi
                # Relative Pressure Balance: % of last 20 days RSI > 70 minus % of last 20 days RSI < 40
                if rsi_series is not None and len(rsi_series.dropna()) >= 20:
                    last_20_rsi = rsi_series.iloc[-20:]
                    valid_last_20_rsi = last_20_rsi.dropna()
                    hot_pct = (valid_last_20_rsi > 70).mean() * 100 if len(valid_last_20_rsi) > 0 else np.nan
                    cold_pct = (valid_last_20_rsi < 40).mean() * 100 if len(valid_last_20_rsi) > 0 else np.nan
                    relative_pressure_balance = hot_pct - cold_pct if np.isfinite(hot_pct) and np.isfinite(cold_pct) else np.nan
                # Momentum Efficiency Slope: RSI slope (last 20d) / log-ratio slope (last 20d), both normalized by std
                if rsi_series is not None and len(rsi_series.dropna()) >= 20 and len(ratio_series.dropna()) >= 20:
                    try:
                        # RSI slope (last 20d)
                        y_rsi = rsi_series.dropna().iloc[-20:].values.reshape(-1,1)
                        x_rsi = np.arange(len(y_rsi)).reshape(-1,1)
                        rsi_std = np.std(y_rsi) if np.std(y_rsi) != 0 else np.nan
                        rsi_slope = np.nan
                        if rsi_std and not np.isnan(rsi_std):
                            model_rsi = LinearRegression().fit(x_rsi, y_rsi)
                            rsi_slope = model_rsi.coef_[0][0] / rsi_std
                        rsi_slope_20d = rsi_slope
                        # Ratio slope (last 20d, log scale)
                        y_ratio = np.log(ratio_series.dropna().iloc[-20:].values.reshape(-1,1))
                        x_ratio = np.arange(len(y_ratio)).reshape(-1,1)
                        ratio_std = np.std(y_ratio) if np.std(y_ratio) != 0 else np.nan
                        ratio_slope = np.nan
                        if ratio_std and not np.isnan(ratio_std):
                            model_ratio = LinearRegression().fit(x_ratio, y_ratio)
                            ratio_slope = model_ratio.coef_[0][0] / ratio_std
                        # Final metric
                        if ratio_slope and not np.isnan(ratio_slope) and rsi_slope and not np.isnan(rsi_slope) and ratio_slope != 0:
                            momentum_efficiency_slope = rsi_slope / ratio_slope
                        else:
                            momentum_efficiency_slope = np.nan
                    except Exception:
                        momentum_efficiency_slope = np.nan
                        rsi_slope_20d = np.nan
            
            # MACD Histogram and Persistence only
            macd_hist_norm = np.nan
            hist_persistence = np.nan
            macd_hist_slope_20d = np.nan
            if ratio_series is not None and len(ratio_series) >= 26:
                ema_12 = ratio_series.ewm(span=12, adjust=False).mean()
                ema_26 = ratio_series.ewm(span=26, adjust=False).mean()
                macd = ema_12 - ema_26
                macd_signal = macd.ewm(span=9, adjust=False).mean()
                macd_hist = macd - macd_signal
                macd_hist_norm = (macd_hist.iloc[-1] / ema_26.iloc[-1]) * 100
                # Histogram Persistence: % of last 20 days normalized macd_hist > 0.0
                if len(macd_hist) >= 20 and np.isfinite(ema_26.iloc[-1]):
                    macd_hist_norm_series = macd_hist / ema_26 * 100
                    last_20_hist_norm = macd_hist_norm_series.iloc[-20:]
                    hist_persistence = (last_20_hist_norm > 0.0).mean() * 100 if len(last_20_hist_norm) > 0 else np.nan
                # MACD Histogram Slope (20d, linear regression, normalized to last value)
                if len(macd_hist) >= 20 and np.isfinite(ema_26.iloc[-1]):
                    try:
                        y_macd_hist = macd_hist.iloc[-20:].values.reshape(-1,1)
                        x_macd_hist = np.arange(len(y_macd_hist)).reshape(-1,1)
                        model_macd_hist = LinearRegression().fit(x_macd_hist, y_macd_hist)
                        slope_macd_hist = model_macd_hist.coef_[0][0]
                        macd_hist_slope_20d = slope_macd_hist / y_macd_hist[-1][0] if y_macd_hist[-1][0] != 0 else np.nan
                    except Exception as e:
                        macd_hist_slope_20d = np.nan
            
            def to_scalar(val):
                if isinstance(val, (pd.Series, pd.DataFrame, np.ndarray, list)):
                    try:
                         if hasattr(val, 'item'):
                             return val.item()
                         if len(val) == 1:
                             return val[0]
                    except:
                        pass
                    return np.nan
                return val
            
            result = {
                'Ticker': ticker,
                'CAGR': to_scalar(cagr),
                'Ratio RS vs VOO 12M ex1': to_scalar(rs_vs_voo_12m_ex1),
                'Ratio RS vs VOO 6M': to_scalar(rs_vs_voo),
                'RS Slope 6M ex1M': to_scalar(rs_slope_6m_ex1m),
                'RS Volatility 6M ex1M': to_scalar(rs_vol_6m_ex1m),
                'RS Volatility 12M ex1M': to_scalar(rs_vol_12m_ex1m),
                'Ratio vs SMA200': to_scalar(ratio_vs_sma200),
                'Ratio SMA200 Slope 6M': to_scalar(ratio_sma200_slope_126d),
                '% Days Above SMA200 6M': to_scalar(pct_above_sma200_6mo),
                'OBV Slope 100D': to_scalar(obv_slope_100d),
                'Persistent RVOL 60D': to_scalar(persistent_rvol_60d),
                'UVP 60D': to_scalar(uvp_60d),
                'UVP 20D': to_scalar(uvp_20d),
                'Distribution Days 30D': to_scalar(distribution_days),
                'Max Drawdown 6M': to_scalar(max_drawdown_6mo),
                'Volatility 1Y': to_scalar(volatility_1y),
                'Volatility Ratio 20D/100D': to_scalar(volatility_ratio),
                'RSI 14D': to_scalar(rsi_current),
                'RSI Slope 20D': to_scalar(rsi_slope_20d),
                'Momentum Efficiency Slope': to_scalar(momentum_efficiency_slope),
                'Relative Pressure Balance 20D': to_scalar(relative_pressure_balance),
                'MACD Histogram': to_scalar(macd_hist_norm),
                'MACD Histogram Slope 20D': to_scalar(macd_hist_slope_20d),
                'Histogram Persistence 20D': to_scalar(hist_persistence),
            }
            results.append(result)
        except Exception as e:
            print(f"  Error processing {ticker}: {str(e)}")
            import traceback
            traceback.print_exc()
            continue
    
    df = pd.DataFrame(results)
    
    if df.empty:
        print("No valid results to display")
        return None
    
    df.set_index('Ticker', inplace=True)
    
    # Define the new column order and border groups
    ordered_cols = [
        'CAGR',
        'Ratio RS vs VOO 12M ex1',
        'Ratio RS vs VOO 6M',
        'RS Slope 6M ex1M',
        'RS Volatility 6M ex1M',
        'RS Volatility 12M ex1M',
        'Ratio vs SMA200',
        'Ratio SMA200 Slope 6M',
        '% Days Above SMA200 6M',
        'OBV Slope 100D',
        'Persistent RVOL 60D',
        'UVP 60D',
        'UVP 20D',
        'Distribution Days 30D',
        'Max Drawdown 6M',
        'Volatility 1Y',
        'Volatility Ratio 20D/100D',
        'RSI 14D',
        'RSI Slope 20D',
        'Momentum Efficiency Slope',
        'Relative Pressure Balance 20D',
        'MACD Histogram',
        'MACD Histogram Slope 20D',
        'Histogram Persistence 20D'
    ]
    
    # Only keep columns that exist in the DataFrame (in case some are missing)
    display_cols = [col for col in ordered_cols if col in df.columns]
    df_display = df[display_cols].copy()
    
    # Round all values in df_display to 2 decimal places (except for non-numeric columns)
    for col in df_display.columns:
        if pd.api.types.is_numeric_dtype(df_display[col]):
            df_display[col] = df_display[col].round(2)
    
    # Define border-right for spacers (after these columns)
    border_after = {
        'RS Volatility 12M ex1M',
        '% Days Above SMA200 6M',
        'Distribution Days 30D',
        'Volatility Ratio 20D/100D',
    }
    
    def apply_column_styles(series):
        col_name = series.name
        styles = [''] * len(series)
        # Add right border if this column is a spacer
        if col_name in border_after:
            styles = ["border-right: 3px solid #333;" for _ in series]
        return styles
    
    def color_by_thresholds(val, col):
        if col not in thresholds or pd.isna(val):
            return ''
        t = thresholds[col]
        thresholds_list = t['thresholds']
        # Use fixed color palette
        colors = ["#2d5732", "#167522", "#20a830", "#45cc56", "#95f5a1"]
        # Use interval logic: (-inf, t1), [t1, t2), [t2, t3), ... [tn, inf)
        category_index = 0
        for i, threshold in enumerate(thresholds_list):
            if val < threshold:
                category_index = i
                break
        else:
            category_index = len(thresholds_list)
        if category_index >= len(colors):
            category_index = len(colors) - 1
        return f'background-color: {colors[category_index]}; color: black;'

    def apply_coloring(styled_df):
        for col in styled_df.data.columns:
            if col in thresholds:
                styled_df = styled_df.apply(
                    lambda s: [color_by_thresholds(v, s.name) for v in s], 
                    axis=0, 
                    subset=[col]
                )
        return styled_df

    styled = df_display.style.apply(apply_column_styles, axis=0)
    format_dict = {col: '{:.2f}' for col in df_display.columns}
    styled = styled.format(format_dict, na_rep='-')
    styled = apply_coloring(styled)
    display(styled)
    return df


In [80]:
import json

def concat_lists_from_json(json_path, list_names):
    """
    Given a JSON file containing a dict of lists, and a list of keys (list_names),
    return a single concatenated list of all lists corresponding to those keys.
    """
    with open(json_path, 'r') as f:
        data = json.load(f)
    result = []
    for name in list_names:
        result.extend(data.get(name, []))
    return result
json_path = 'C:\\Users\\wongb\\compounding-focus-finance-tooling\\compounding-focus-finance-tooling\\holdings.json'
etf_list = ['nuclear']
etf_tickers = concat_lists_from_json(json_path, etf_list)

In [81]:
# Example usage
baseline_tickers = ['VOO']
held_tickers = ['SGI', 'GS', 'HSBC', 'IAU', 'SIVR', 'AMD', 'INTC', 'VRT']
potential_tickers = ['CFG', 'VSAT', 'ASTS', 'NXST', 'EL', 'HSBC', 'JNJ', 'MNST', 'GLW', 'SEB', 'ADM', 'CVS', 'LDOS', 'CMI']
colder_tickers = ['TEL', 'GOOG', 'MS', 'GE', 'DDS', 'EA', 'IDXX', 'INSM', 'FER', 'ROST', 'FLEX', 'COHR', 'MDB']
recovering_tickers = ['GILD', 'COR', 'CRUS', 'JBL', 'MCHP', 'BKR', 'CELH', 'BG', 'CASY', 'ENTG']
hotter_tickers = ['STX', 'WDC', 'ASML', 'ADI', 'KLAC', 'LRCX', 'AMAT', 'MU', 'ULTA', 'GDX', 'AMKR', 'MTSI', 'ALAB', 'TER', 'SNDK', 'BAP', 'BBD']
test_tickers = ['SETM', 'REMX', 'XME', 'IAU', 'URA', 'GDX', 'SIL', 'COPX', 'URNM',  'NLR','URNM', 'SMH', 'SOXX', 'AIRR', 'ICOP']
tickers = baseline_tickers + held_tickers + etf_tickers
from collections import Counter
# Remove all duplicates, keep only one occurrence of each ticker
tickers = list(dict.fromkeys(tickers))
dupes = [item for item, count in Counter(tickers).items() if count > 1]
if dupes:
    print(f"Duplicate tickers found: {dupes}")
else:
    df = momentum_screener(tickers)

Fetching benchmark data (VOO)...


,CAGR,Ratio RS vs VOO 12M ex1,Ratio RS vs VOO 6M,RS Slope 6M ex1M,RS Volatility 6M ex1M,RS Volatility 12M ex1M,Ratio vs SMA200,Ratio SMA200 Slope 6M,% Days Above SMA200 6M,OBV Slope 100D,Persistent RVOL 60D,UVP 60D,UVP 20D,Distribution Days 30D,Max Drawdown 6M,Volatility 1Y,Volatility Ratio 20D/100D,RSI 14D,RSI Slope 20D,Momentum Efficiency Slope,Relative Pressure Balance 20D,MACD Histogram,MACD Histogram Slope 20D,Histogram Persistence 20D
Ticker,,,,,,,,,,,,,,,,,,,,,,,,
VOO,14.72,-0.00,-0.00,-0.00,0.00,0.00,-0.00,0.00,14.29,0.39,26.67,18.22,9.32,2.00,5.06,18.40,0.61,-,-0.00,-,-,0.00,-0.17,100.00
SGI,15.88,38.02,15.20,0.11,28.26,27.47,9.88,0.13,100.00,-0.00,23.33,6.93,6.38,0.00,10.29,33.25,0.51,58.88,0.12,0.88,-25.00,0.28,0.21,50.00
GS,11.62,39.32,18.12,0.08,18.19,18.77,16.36,0.09,100.00,-0.21,23.33,19.36,-6.62,6.00,7.78,31.45,1.20,48.93,0.01,0.08,30.00,-0.34,-0.01,55.00
HSBC,6.25,45.80,19.94,0.07,20.06,20.34,18.08,0.11,100.00,0.78,26.67,26.39,44.31,0.00,9.06,25.00,0.87,66.22,-0.13,-0.86,60.00,0.10,-0.28,65.00
IAU,12.10,44.96,35.08,0.15,21.12,26.45,23.09,0.11,100.00,0.04,21.67,20.16,31.28,2.00,10.05,20.30,0.83,88.85,0.08,0.53,30.00,0.98,0.04,60.00
SIVR,12.52,105.49,139.66,0.26,31.87,30.10,105.38,0.16,100.00,0.62,43.33,27.30,26.66,8.00,13.63,35.96,0.95,78.66,-0.04,-0.24,45.00,1.54,0.02,100.00
AMD,10.10,51.63,46.56,0.31,57.44,50.13,41.07,0.18,100.00,0.05,21.67,2.30,40.45,0.00,25.05,60.95,0.48,70.97,0.14,1.18,5.00,2.42,0.03,85.00
INTC,12.78,59.97,82.23,0.45,58.27,57.56,44.07,0.16,88.10,0.14,35.00,11.30,24.18,4.00,19.05,68.40,0.99,57.71,0.14,0.88,25.00,1.53,0.14,75.00
VRT,47.58,20.98,27.67,0.24,46.25,59.43,21.60,0.10,99.21,-0.00,16.67,-6.63,8.73,3.00,24.78,68.94,0.56,56.23,0.14,1.18,-25.00,1.11,0.06,75.00


In [28]:
# Quick summary stat sentence for a given ticker

def print_summary_stat(df, ticker):

    if ticker not in df.index:
        print(f"Ticker {ticker} not found in DataFrame.")
        return
    row = df.loc[ticker]
    summary = []
    two_decimal_cols = [
        '% Days Above SMA200 6M',
        'MACD (ratio, %)',
        'MACD Histogram',
        'MACD Histogram Slope 20D',
        'Volatility Ratio 20D/100D',
        'OBV Slope 100D',
        'RS Slope 6M ex1M',
        'Ratio SMA200 Slope 50D',
        'Momentum Efficiency Slope',
        'RSI Slope 20D'
    ]  # 'RSI Up/Down Volatility Ratio' removed
    for col, val in row.items():
        # Remove anything in parentheses from the column name for display
        col_clean = col.split('(')[0].strip()
        # Format value
        if isinstance(val, float):
            if col in two_decimal_cols:
                val_str = f"{val:.2f}"
            else:
                val_str = f"{int(round(val))}"
        else:
            val_str = str(val)
        summary.append(f"{col_clean}: {val_str}")
    print(f"Summary for {ticker}: " + "; ".join(summary))


In [83]:
stock = 'NLR'
print_summary_stat(df, stock)

Summary for NLR: CAGR: 5; Ratio RS vs VOO 12M ex1: 35; Ratio RS vs VOO 6M: 19; RS Slope 6M ex1M: 0.11; RS Volatility 6M ex1M: 35; RS Volatility 12M ex1M: 34; Ratio vs SMA200: 22; Ratio SMA200 Slope 6M: 0; % Days Above SMA200 6M: 100.00; OBV Slope 100D: 0.33; Persistent RVOL 60D: 23; UVP 60D: 14; UVP 20D: 34; Distribution Days 30D: 1; Max Drawdown 6M: 26; Volatility 1Y: 40; Volatility Ratio 20D/100D: 0.80; RSI 14D: 84; RSI Slope 20D: 0.16; Momentum Efficiency Slope: 0.96; Relative Pressure Balance 20D: 35; MACD Histogram: 1.17; MACD Histogram Slope 20D: 0.06; Histogram Persistence 20D: 100
